# ResNet-UNet Model Training

## Import Libraries and Shared Utilities

In [3]:
import sys
import os
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path to import utils
parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f"Utils path: {parent_dir}")
print(f"Utils file exists: {os.path.exists(os.path.join(parent_dir, 'utils.py'))}\n")

# Import shared utility functions
try:
    from utils import (
        load_image_dataset,
        normalize,
        augment_image,
        load_dataset_with_info,
        BASE_PATH,
        IMAGE_SIZE
    )
    print(f"✅ Successfully imported from utils.py")
except ImportError as e:
    print(f"❌ Import Error: {e}")
    print(f"   Current directory: {Path.cwd()}")
    print(f"   Parent directory: {parent_dir}")
    print(f"   Python path: {sys.path[:3]}...")
    raise


Utils path: c:\Users\Paudel\Desktop\techsprint_xhack\ml-app
Utils file exists: True

✅ Successfully imported from utils.py


## Load and Prepare Datasets

Using shared `load_dataset_with_info` function from utils.py

In [4]:
# Configuration
BATCH_SIZE = 16
IMAGE_SIZE = 224

# Enable memory growth for GPU to prevent OOM
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"⚠️ GPU config error: {e}")

# Clear any existing models from memory
tf.keras.backend.clear_session()
import gc
gc.collect()

print(f"Dataset Base Path: {BASE_PATH}")
print(f"Path exists: {BASE_PATH.exists()}\n")

# ============================================================
# LAZY LOADING: Generator-based dataset that loads images on-demand
# ============================================================
from PIL import Image

def create_lazy_dataset(split, category, base_path, batch_size=16, image_size=224):
    """
    Creates a tf.data.Dataset that lazily loads images batch-by-batch.
    Images are only loaded into memory when needed, preventing OOM errors.
    """
    full_path = Path(base_path) / split / category
    
    if not full_path.exists():
        print(f"❌ {split}/{category}: Path not found - {full_path}")
        return None, 0
    
    # Get file paths (not images themselves - lazy!)
    valid_extensions = ['.tif', '.jpg', '.png', '.jpeg', '.bmp', '.gif']
    file_paths = [str(f) for f in full_path.glob('*') if f.suffix.lower() in valid_extensions]
    file_count = len(file_paths)
    
    print(f"✅ {split:5s}/{category:4s}: {file_count:4d} images (lazy loading)")
    
    def image_generator():
        """Generator that yields one image at a time"""
        for file_path in file_paths:
            try:
                with Image.open(file_path) as img:
                    # Convert to RGB (handles grayscale/RGBA)
                    if category == 'Mask':
                        img = img.convert('L')  # Grayscale for masks
                        img = img.resize((image_size, image_size))
                        arr = np.array(img, dtype=np.float32) / 255.0
                        arr = np.expand_dims(arr, axis=-1)  # (H, W, 1)
                    else:
                        img = img.convert('RGB')
                        img = img.resize((image_size, image_size))
                        arr = np.array(img, dtype=np.float32) / 255.0  # (H, W, 3)
                    yield arr
            except Exception as e:
                print(f"⚠️ Skipping {file_path}: {e}")
                continue
    
    # Define output shape based on category
    if category == 'Mask':
        output_sig = tf.TensorSpec(shape=(image_size, image_size, 1), dtype=tf.float32)
    else:
        output_sig = tf.TensorSpec(shape=(image_size, image_size, 3), dtype=tf.float32)
    
    # Create dataset from generator
    dataset = tf.data.Dataset.from_generator(
        image_generator,
        output_signature=output_sig
    )
    
    # Batch and prefetch (images loaded only when batch is requested)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=2)  # Only prefetch 2 batches
    
    return dataset, file_count

# Load datasets lazily (no images in memory yet!)
print("\n📦 Creating LAZY datasets (images load on-demand):\n")

print("Training Set:")
train_au_ds, train_au_count = create_lazy_dataset('train', 'Au', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
train_tp_ds, train_tp_count = create_lazy_dataset('train', 'Tp', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
train_mask_ds, train_mask_count = create_lazy_dataset('train', 'Mask', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)

print("\nTest Set:")
test_au_ds, _ = create_lazy_dataset('test', 'Au', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
test_tp_ds, _ = create_lazy_dataset('test', 'Tp', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
test_mask_ds, _ = create_lazy_dataset('test', 'Mask', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)

print("\nValidation Set:")
val_au_ds, _ = create_lazy_dataset('val', 'Au', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
val_tp_ds, _ = create_lazy_dataset('val', 'Tp', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
val_mask_ds, _ = create_lazy_dataset('val', 'Mask', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)

print("\n" + "="*60)
print("✅ ALL LAZY DATASETS CREATED!")
print("   Images will load batch-by-batch during training")
print("="*60)


Dataset Base Path: c:\Users\Paudel\Desktop\techsprint_xhack\ml-app\split_dataset
Path exists: True


📦 Creating LAZY datasets (images load on-demand):

Training Set:
✅ train/Au  : 1181 images (lazy loading)
✅ train/Tp  : 3151 images (lazy loading)
✅ train/Mask: 3151 images (lazy loading)

Test Set:
✅ test /Au  :  337 images (lazy loading)
✅ test /Tp  :  839 images (lazy loading)
✅ test /Mask:  839 images (lazy loading)

Validation Set:
✅ val  /Au  :  170 images (lazy loading)
✅ val  /Tp  :  445 images (lazy loading)
✅ val  /Mask:  445 images (lazy loading)

✅ ALL LAZY DATASETS CREATED!
   Images will load batch-by-batch during training


## Normalization of images

In [5]:
# Images are already normalized (0-1) in the lazy loader
# No need to normalize again - just verify

print("✅ Images already normalized in lazy loader (divided by 255)")
print("   Skipping separate normalization step to save memory\n")

# Verify by checking a sample batch
print("🔍 Verifying sample batch:")
for batch in train_au_ds.take(1):
    print(f"   Shape: {batch.shape}")
    print(f"   Value range: [{batch.numpy().min():.3f}, {batch.numpy().max():.3f}]")
    print(f"   Dtype: {batch.dtype}")

✅ Images already normalized in lazy loader (divided by 255)
   Skipping separate normalization step to save memory

🔍 Verifying sample batch:
   Shape: (16, 224, 224, 3)
   Value range: [0.000, 1.000]
   Dtype: <dtype: 'float32'>


## Data Argumentation

In [6]:
# Light augmentation applied per-batch (memory efficient)
def augment_batch(images):
    """Apply augmentation to a batch of images"""
    # Random horizontal flip
    images = tf.image.random_flip_left_right(images)
    # Random brightness
    images = tf.image.random_brightness(images, max_delta=0.2)
    images = tf.clip_by_value(images, 0.0, 1.0)
    # Random contrast
    images = tf.image.random_contrast(images, lower=0.8, upper=1.2)
    images = tf.clip_by_value(images, 0.0, 1.0)
    return images

# Apply augmentation to training datasets (lazy - applied when batch is fetched)
train_au_ds = train_au_ds.map(augment_batch, num_parallel_calls=tf.data.AUTOTUNE)
train_tp_ds = train_tp_ds.map(augment_batch, num_parallel_calls=tf.data.AUTOTUNE)

print('✅ Augmentation applied (will execute lazily per batch)')

✅ Augmentation applied (will execute lazily per batch)


## Create Properly Paired Tp-Mask Dataset for Segmentation

The mask filename = `{tampered_filename_without_extension}_gt.png`. We need explicit filename-based pairing to ensure each tampered image is matched with its correct ground truth mask.

In [7]:
import glob
from pathlib import Path
from PIL import Image
import numpy as np

def create_paired_tp_mask_dataset(split, base_path, batch_size=32, image_size=224):
   
    tp_dir = base_path / split / 'Tp'
    mask_dir = base_path / split / 'Mask'
    
    # Get all tampered image files
    tp_files = list(tp_dir.glob('*'))
    tp_files = [f for f in tp_files if f.suffix.lower() in ['.tif', '.jpg', '.png', '.jpeg', '.bmp']]
    
    paired_tp_paths = []
    paired_mask_paths = []
    missing_masks = []
    
    for tp_file in tp_files:
        # Construct expected mask filename: {basename}_gt.png
        tp_basename = tp_file.stem  # filename without extension
        expected_mask_name = f"{tp_basename}_gt.png"
        mask_path = mask_dir / expected_mask_name
        
        if mask_path.exists():
            paired_tp_paths.append(str(tp_file))
            paired_mask_paths.append(str(mask_path))
        else:
            missing_masks.append(tp_file.name)
    
    print(f"\n📁 {split.upper()} Paired Dataset:")
    print(f"   ✅ Successfully paired: {len(paired_tp_paths)} images")
    if missing_masks:
        print(f"   ⚠️  Missing masks for {len(missing_masks)} images")
        if len(missing_masks) <= 5:
            for m in missing_masks:
                print(f"      - {m}")
    
    # Use Python generator with PIL - with proper resource cleanup
    def image_generator():
        for tp_path, mask_path in zip(paired_tp_paths, paired_mask_paths):
            try:
                # Load tampered image with PIL (handles TIFF) - use context manager
                with Image.open(tp_path) as tp_img:
                    tp_img = tp_img.convert('RGB')
                    tp_img = tp_img.resize((image_size, image_size))
                    tp_arr = np.array(tp_img, dtype=np.float32) / 255.0
                
                # Load mask with PIL - use context manager
                with Image.open(mask_path) as mask_img:
                    mask_img = mask_img.convert('L')  # Grayscale
                    mask_img = mask_img.resize((image_size, image_size))
                    mask_arr = np.array(mask_img, dtype=np.float32) / 255.0
                    mask_arr = np.expand_dims(mask_arr, axis=-1)  # Add channel dim
                
                yield tp_arr, mask_arr
            except Exception as e:
                print(f"   ⚠️  Skipping {tp_path}: {e}")
                continue
    
    # Create dataset from generator
    dataset = tf.data.Dataset.from_generator(
        image_generator,
        output_signature=(
            tf.TensorSpec(shape=(image_size, image_size, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(image_size, image_size, 1), dtype=tf.float32)
        )
    )
    
    # Use smaller shuffle buffer to reduce memory usage
    dataset = dataset.shuffle(buffer_size=min(200, len(paired_tp_paths)))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset, len(paired_tp_paths)

# Create paired datasets for Stage 2 segmentation training
print("="*60)
print("CREATING PAIRED Tp-MASK DATASETS FOR SEGMENTATION")
print("="*60)

train_paired_ds, train_count = create_paired_tp_mask_dataset('train', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
val_paired_ds, val_count = create_paired_tp_mask_dataset('val', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
test_paired_ds, test_count = create_paired_tp_mask_dataset('test', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)

print("\n" + "="*60)
print("✅ PAIRED DATASETS CREATED SUCCESSFULLY!")
print(f"   Train: {train_count} | Val: {val_count} | Test: {test_count}")
print("="*60)

# Verify a sample batch
print("\n🔍 Verifying sample batch shapes:")
for tp_batch, mask_batch in train_paired_ds.take(1):
    print(f"   Tampered images: {tp_batch.shape}")
    print(f"   Mask images: {mask_batch.shape}")
    print(f"   Tp value range: [{tp_batch.numpy().min():.3f}, {tp_batch.numpy().max():.3f}]")
    print(f"   Mask value range: [{mask_batch.numpy().min():.3f}, {mask_batch.numpy().max():.3f}]")

CREATING PAIRED Tp-MASK DATASETS FOR SEGMENTATION

📁 TRAIN Paired Dataset:
   ✅ Successfully paired: 3151 images

📁 VAL Paired Dataset:
   ✅ Successfully paired: 445 images

📁 TEST Paired Dataset:
   ✅ Successfully paired: 839 images

✅ PAIRED DATASETS CREATED SUCCESSFULLY!
   Train: 3151 | Val: 445 | Test: 839

🔍 Verifying sample batch shapes:
   Tampered images: (16, 224, 224, 3)
   Mask images: (16, 224, 224, 1)
   Tp value range: [0.000, 1.000]
   Mask value range: [0.000, 1.000]


## Training ResNet-50 Classifier (Authentic vs Tampered)



In [8]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


# Load ResNet-50 pretrained on ImageNet
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

print(f"\n ResNet-50 Info:")
print(f"   Total layers: {len(base_model.layers)}")

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze last 30 layers for fine-tuning
UNFREEZE_LAYERS = 30
for layer in base_model.layers[-UNFREEZE_LAYERS:]:
    layer.trainable = True

trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
frozen_count = len(base_model.layers) - trainable_count
print(f"   Frozen layers: {frozen_count}")
print(f"   Trainable layers: {trainable_count}")

# Add classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(1, activation='sigmoid')(x)  # Binary: Authentic (0) vs Tampered (1)

classifier_model = Model(inputs=base_model.input, outputs=x, name='ResNet50_Classifier')

print(f"\n✅ Classifier built!")
print(f"   Output: Binary (0=Authentic, 1=Tampered)")
print(f"   Total params: {classifier_model.count_params():,}")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step

 ResNet-50 Info:
   Total layers: 175
   Frozen layers: 145
   Trainable layers: 30

✅ Classifier built!
   Output: Binary (0=Authentic, 1=Tampered)
   Total params: 24,776,577


##  Classification Data 

In [1]:
# Label datasets: Au=0 (Authentic), Tp=1 (Tampered)
# Unbatch -> add labels -> combine -> shuffle -> rebatch

train_au_labeled = train_au_ds.unbatch().map(lambda x: (x, 0), num_parallel_calls=tf.data.AUTOTUNE)
train_tp_labeled = train_tp_ds.unbatch().map(lambda x: (x, 1), num_parallel_calls=tf.data.AUTOTUNE)

val_au_labeled = val_au_ds.unbatch().map(lambda x: (x, 0), num_parallel_calls=tf.data.AUTOTUNE)
val_tp_labeled = val_tp_ds.unbatch().map(lambda x: (x, 1), num_parallel_calls=tf.data.AUTOTUNE)

# Combine authentic and tampered
train_ds = train_au_labeled.concatenate(train_tp_labeled)
val_ds = val_au_labeled.concatenate(val_tp_labeled)

# Shuffle with small buffer (memory efficient), batch, prefetch only 2 batches
train_ds = train_ds.shuffle(200).batch(BATCH_SIZE).prefetch(2)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(2)

print("✅ Training data: Authentic + Tampered images with labels (lazy)")
print("✅ Validation data prepared (lazy)")

# Compile classifier
classifier_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled with binary crossentropy")

# Train Stage 1
EPOCHS = 12

print(f"\n🚀 STAGE 1: Training classifier for {EPOCHS} epochs...")
print("="*60)

# Clear memory before training
gc.collect()

history = classifier_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)

print(f"\nFinal Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

# Save the trained encoder for Stage 2
base_model.save('resnet50_encoder_trained.keras')

# Free memory
gc.collect()

NameError: name 'train_au_ds' is not defined

## UNet Decoder for Mask Prediction

In [ ]:
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, Concatenate, Activation
import tensorflow.keras.backend as K


# Freeze the trained encoder completely
for layer in base_model.layers:
    layer.trainable = False

print(f"✅ Encoder frozen (all {len(base_model.layers)} layers)")

# Get skip connections from encoder
skip_layer_names = ['conv1_relu', 'conv2_block3_out', 'conv3_block4_out', 'conv4_block6_out']
skip_connections = [base_model.get_layer(name).output for name in skip_layer_names]
encoder_output = base_model.output

# Custom loss functions
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

# Decoder block
def decoder_block(inputs, skip_features, num_filters):
    x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Concatenate()([x, skip_features])
    x = Conv2D(num_filters, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Conv2D(num_filters, (3, 3), padding='same', activation='relu')(x)
    return x

# Build decoder
x = encoder_output
x = Conv2DTranspose(512, (2, 2), strides=2, padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Concatenate()([x, skip_connections[3]])
x = Conv2D(512, (3, 3), padding='same', activation='relu')(x)

x = decoder_block(x, skip_connections[2], 256)
x = decoder_block(x, skip_connections[1], 128)
x = decoder_block(x, skip_connections[0], 64)

x = Conv2DTranspose(32, (2, 2), strides=2, padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Conv2D(32, (3, 3), padding='same', activation='relu')(x)

output = Conv2D(1, (1, 1), activation='sigmoid', name='mask_output')(x)

unet_model = Model(inputs=base_model.input, outputs=output, name='ResNet50_UNet')

# Count trainable params (only decoder)
trainable_params = sum([K.count_params(w) for w in unet_model.trainable_weights])
print(f"\n✅ UNet built with trained encoder!")
print(f"   Trainable params (decoder only): {trainable_params:,}")
print(f"   Output shape: {unet_model.output_shape}")

## Train Stage 2: UNet Decoder with Mask Data

In [ ]:
# ============================================================
# STAGE 2: TRAINING UNET DECODER WITH PROPERLY PAIRED DATA
# ============================================================

print("="*60)
print("STAGE 2: TRAINING UNET WITH PAIRED Tp-MASK DATA")
print("="*60)

# The train_paired_ds and val_paired_ds were created earlier
# with proper filename-based pairing (Tp + corresponding _gt.png mask)

def prepare_mask(mask):
    """Ensure mask is binary (0 or 1)"""
    if len(mask.shape) == 3 and mask.shape[-1] == 3:
        mask = tf.reduce_mean(mask, axis=-1, keepdims=True)
    mask = tf.cast(mask > 0.5, tf.float32)
    return mask

# Apply mask preparation to paired datasets
train_seg_ds = train_paired_ds.map(
    lambda img, mask: (img, prepare_mask(mask)), 
    num_parallel_calls=tf.data.AUTOTUNE
)
val_seg_ds = val_paired_ds.map(
    lambda img, mask: (img, prepare_mask(mask)), 
    num_parallel_calls=tf.data.AUTOTUNE
)

print("✅ Using properly paired Tp-Mask datasets (filename-matched)")

# Compile UNet
unet_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=bce_dice_loss,
    metrics=['binary_accuracy', dice_coefficient]
)

print("✅ UNet compiled with BCE+Dice loss")

# Train Stage 2
EPOCHS_STAGE2 = 12

print(f"\n🚀 STAGE 2: Training decoder for {EPOCHS_STAGE2} epochs...")
print("="*60)

history_seg = unet_model.fit(
    train_seg_ds,
    validation_data=val_seg_ds,
    epochs=EPOCHS_STAGE2,
    verbose=1
)

print("\n" + "="*60)
print("✅ STAGE 2 COMPLETE!")
print("="*60)

# Save the complete model
unet_model.save('resnet50_unet_tampering_detector.keras')
print("\n📁 Complete UNet model saved: resnet50_unet_tampering_detector.keras")